<a href="https://colab.research.google.com/github/arjunnamburi/ml_death_overs_ipl/blob/main/notebooks/results_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**2025 validation scores of the final models, written to a file (added Nov 16, after the test).** It reads the 2025 predictions that `models.ipynb` saved before the freeze (`bundle_2025.parquet`: XGBoost `px_*`, the vector-scaled logistic regression `pl_*`, B1 `pb1_*`) and recomputes log-loss, 95% match-bootstrap intervals and Brier, plus the paired differences used to choose the models. Nothing is refit, and it runs on a CPU in a few seconds. B0 on 2025 is already in `data/results.csv`. Check: the log-losses must equal the `logloss 2025` column of `data/results_2026.csv`.

In [1]:
# ===== V0. Setup (COPY-PASTE) =====
import os
import numpy as np
import pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA = '/content/drive/MyDrive/IPL PROJECT ALL DOCS/data/'
except ImportError:                                   # outside Colab: point IPL_DATA at a folder holding bundle_2025.parquet
    DATA = os.environ['IPL_DATA']

b = pd.read_parquet(DATA + 'bundle_2025.parquet')
y, match, death = b['y'].to_numpy(), b['match_id'].to_numpy(), b['death'].to_numpy()
MODELS = {'X-hist': 'px', 'B2-hist +V': 'pl', 'B1': 'pb1'}
P = {m: b[[f'{p}_{k}' for k in range(7)]].to_numpy() for m, p in MODELS.items()}
print(b.shape, int(death.sum()), b['match_id'].nunique())   # expect (15809, 34) 3466 69

(15809, 34) 3466 69


In [2]:
# ===== V1. Scores and paired differences on 2025 (COPY-PASTE) =====
EPS = 1e-15
def per_ball_logloss(P, y): return -np.log(np.clip(P[np.arange(len(y)), y], EPS, 1))
def per_ball_brier(P, y):
    O = np.zeros_like(P); O[np.arange(len(y)), y] = 1
    return ((P - O) ** 2).sum(axis=1)

def boot(x, mask, n_boot=2000, seed=0):
    """Mean over the slice and its 95% match-bootstrap interval (the same resampling as slice_paired)."""
    codes, _ = pd.factorize(match[mask])
    s, c = np.bincount(codes, weights=x[mask]), np.bincount(codes)
    idx = np.random.default_rng(seed).integers(0, len(s), size=(n_boot, len(s)))
    bt = s[idx].sum(axis=1) / c[idx].sum(axis=1)
    return x[mask].mean(), *np.percentile(bt, [2.5, 97.5])

SLICES = {'all': np.ones(len(y), bool), 'death': death}
L = {m: per_ball_logloss(p, y) for m, p in P.items()}
B = {m: per_ball_brier(p, y) for m, p in P.items()}

rows = []
for m in MODELS:
    for sl, mask in SLICES.items():
        mean, lo, hi = boot(L[m], mask)
        rows.append(dict(model=m, split='val', slice=sl, balls=int(mask.sum()), logloss=mean, ll_low=lo, ll_high=hi, brier=B[m][mask].mean()))
res = pd.DataFrame(rows)

rows = []
for a, bb in [('X-hist', 'B2-hist +V'), ('X-hist', 'B1'), ('B2-hist +V', 'B1')]:
    for sl, mask in SLICES.items():
        mean, lo, hi = boot(L[a] - L[bb], mask)
        rows.append(dict(A=a, B=bb, slice=sl, diff=mean, low=lo, high=hi,
                         verdict='A better' if hi < 0 else ('B better' if lo > 0 else "can't tell")))
paired = pd.DataFrame(rows)
print(res.round(4).to_string(index=False)); print(); print(paired.round(4).to_string(index=False))

     model split slice  balls  logloss  ll_low  ll_high  brier
    X-hist   val   all  15809   1.4727  1.4582   1.4878 0.7120
    X-hist   val death   3466   1.5590  1.5334   1.5837 0.7411
B2-hist +V   val   all  15809   1.4762  1.4621   1.4909 0.7137
B2-hist +V   val death   3466   1.5722  1.5464   1.5967 0.7460
        B1   val   all  15809   1.5119  1.4964   1.5292 0.7253
        B1   val death   3466   1.6018  1.5773   1.6260 0.7537

         A          B slice    diff     low    high  verdict
    X-hist B2-hist +V   all -0.0035 -0.0056 -0.0014 A better
    X-hist B2-hist +V death -0.0131 -0.0195 -0.0067 A better
    X-hist         B1   all -0.0393 -0.0454 -0.0337 A better
    X-hist         B1 death -0.0427 -0.0563 -0.0300 A better
B2-hist +V         B1   all -0.0357 -0.0417 -0.0306 A better
B2-hist +V         B1 death -0.0296 -0.0417 -0.0182 A better


In [3]:
# ===== V2. Check against the frozen numbers, then save (COPY-PASTE) =====
r26 = pd.read_csv(DATA + 'results_2026.csv')                 # its "logloss 2025" column was written before the 2026 score
for _, r in res.iterrows():
    frozen = r26.loc[(r26['model'] == r['model']) & (r26['slice'] == r['slice']), 'logloss 2025'].item()
    assert abs(frozen - r['logloss']) < 5e-5, (r['model'], r['slice'], frozen, r['logloss'])
print('matches results_2026.csv (logloss 2025) for all', len(res), 'rows')
res.to_csv(DATA + 'results_2025.csv', index=False)
paired.to_csv(DATA + 'paired_2025.csv', index=False)
print('saved results_2025.csv and paired_2025.csv')

matches results_2026.csv (logloss 2025) for all 6 rows
saved results_2025.csv and paired_2025.csv
